# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

print(np.__version__)
print(scipy.__version__)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

# Find the folder containing the CSV files
matches = list(
    (Path.home() / "Desktop").rglob("p1a_extinction.csv")
)

if not matches:
    raise FileNotFoundError(
        "Cannot find p1a_extinction.csv on your Desktop. "
        "Download the CSV files and put them in the midterm folder."
    )

DATA = matches[0].parent
print("Data folder:", DATA)

def load(name):
    return pd.read_csv(DATA / f"{name}.csv")


## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here

d = load("p1a_extinction")

x = d["airmass"].to_numpy()
y = d["zeropoint_mag"].to_numpy()
sig = d["sigma_quoted_mag"].to_numpy()

A = np.column_stack([x, np.ones(len(x))])
W = 1 / sig**2

# Formal weighted least-squares covariance
formal_cov = np.linalg.inv(A.T @ (W[:, None] * A))
k, m0 = formal_cov @ (A.T @ (W * y))
formal_k_err = np.sqrt(formal_cov[0, 0])

# Compare against np.polyfit
poly_params, poly_cov = np.polyfit(
    x, y, 1, w=1/sig, cov=True
)
poly_k_err = np.sqrt(poly_cov[0, 0])

# Chi-square calculation
resid = y - (k*x + m0)
chi2 = np.sum((resid / sig)**2)
dof = len(x) - 2

p_lower = stats.chi2.cdf(chi2, dof)
p_upper = stats.chi2.sf(chi2, dof)

print(f"k = {k:.6f} +/- {formal_k_err:.6f}")
print(f"polyfit uncertainty = {poly_k_err:.6f}")
print(f"chi2 = {chi2:.4f}")
print(f"reduced chi2 = {chi2/dof:.4f}")
print(f"Lower-tail probability = {p_lower:.4f}")
print(f"Upper-tail probability = {p_upper:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(x, y, yerr=sig, fmt="o",
            capsize=2, label="Observations")

xx = np.linspace(x.min(), x.max(), 100)
ax.plot(xx, k*xx + m0, label="WLS fit")

ax.set_xlabel("Airmass")
ax.set_ylabel("Zero point (mag)")
ax.legend()
plt.tight_layout()
plt.show()


**ANSWER:**

The weighted least-squares fit gives an extinction coefficient of **k = 0.2339 ± 0.0115 mag/airmass**, using the quoted measurement uncertainties.

Using `np.polyfit` with `cov=True`, I get a smaller uncertainty of 0.0069 mag/airmass. This happens because `polyfit` rescales the covariance matrix by the reduced chi-square, which is 0.3582 in this case.

The chi-square is 4.66 with 13 degrees of freedom. The probability of getting a chi-square this small or smaller is only 1.8%, while the upper-tail probability is 98.2%.

This suggests that the pipeline uncertainties may be overestimated, since the data scatter less than expected. I would report **0.0115 mag/airmass** as the uncertainty because it comes directly from the quoted errors without rescaling them based on this particular dataset.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here

# Part 1b: Centroid uncertainty

d = load("p1b_centroid").iloc[0]

psf_sigma = d["psf_sigma_arcsec"]
N = int(d["n_photons"])
quoted = d["quoted_sigma_x0_arcsec"]

# Cramer-Rao lower bound
crlb = psf_sigma / np.sqrt(N)

print(f"PSF sigma = {psf_sigma:.4f} arcsec")
print(f"Number of photons = {N}")
print(f"CRLB = {crlb:.6f} arcsec")
print(f"Quoted uncertainty = {quoted:.6f} arcsec")
print(f"Quoted / CRLB = {quoted/crlb:.4f}")


**ANSWER:**

**ANSWER:**

The reported centroid uncertainty is too small. Using the Cramér–Rao lower bound, I calculated a minimum uncertainty of 0.1101 arcsec, while the classmate reported only 0.06286 arcsec.

The Cramér–Rao bound tells us the smallest possible variance for an unbiased estimator under the assumed model. Since the reported uncertainty is below this limit, it is not consistent with an unbiased estimator under the given assumptions.

One possible reason is that the classmate incorrectly counted some photons more than once or underestimated the measurement noise.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here

# Part 1c: Transit depth uncertainty

d = load("p1c_profile")

depth = d["depth_ppt"].to_numpy()
neg2logL = d["minus2lnL"].to_numpy()

i = int(np.argmin(neg2logL))

# Fit a quadratic near the likelihood minimum
coefs = np.polyfit(
    depth[i-3:i+4] - depth[i],
    neg2logL[i-3:i+4],
    2
)

a, b, c = coefs

best_depth = depth[i] - b / (2*a)
curvature = 2*a

# Correct and incorrect uncertainty
sigma_correct = np.sqrt(2 / curvature)
sigma_classmate = 1 / np.sqrt(curvature)

# Find Delta(-2 ln L) = 1 interval
threshold = neg2logL.min() + 1

lo = np.interp(
    threshold,
    neg2logL[:i+1][::-1],
    depth[:i+1][::-1]
)

hi = np.interp(
    threshold,
    neg2logL[i:],
    depth[i:]
)

print(f"Best depth = {best_depth:.5f} ppt")
print(f"Curvature = {curvature:.5f}")
print(f"Correct uncertainty = {sigma_correct:.5f} ppt")
print(f"Classmate uncertainty = {sigma_classmate:.5f} ppt")
print(f"68% interval = [{lo:.5f}, {hi:.5f}]")

# Monte Carlo coverage simulation
rng = np.random.default_rng(457)

truth = best_depth
n_sim = 20000

fake_estimates = rng.normal(
    truth, sigma_correct, size=n_sim
)

covered_bad = np.mean(
    np.abs(fake_estimates - truth) <= sigma_classmate
)

covered_good = np.mean(
    np.abs(fake_estimates - truth) <= sigma_correct
)

print(f"Bad coverage = {covered_bad:.3%}")
print(f"Correct coverage = {covered_good:.3%}")

# Plot likelihood profile
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    depth,
    neg2logL - neg2logL.min(),
    "o-",
    markersize=3,
    label="Likelihood profile"
)

ax.axhline(
    1,
    linestyle="--",
    color="gray",
    label="Delta(-2 ln L) = 1"
)

ax.axvline(
    best_depth,
    linestyle=":",
    color="black",
    label="Best fit"
)

ax.axvspan(lo, hi, alpha=0.15, label="68% interval")

ax.set_xlabel("Transit depth (ppt)")
ax.set_ylabel("Delta(-2 ln L)")
ax.legend()

plt.tight_layout()
plt.show()


**ANSWER:**

The best-fit transit depth is 3.2035 ppt, with a correct uncertainty of approximately 0.2429 ppt.

The classmate's uncertainty of 0.1717 ppt is too small because they missed a factor of 2 when converting the curvature of the negative twice log-likelihood into an uncertainty. The correct uncertainty is larger by a factor of about 1.414.

I also checked the likelihood profile using a change of 1 in negative twice log-likelihood. This gives an approximate 68% interval from 2.9592 to 3.4448 ppt.

The Monte Carlo test shows that the classmate's interval only covers the true value about 52% of the time, instead of the expected 68%. This confirms that the reported error bar is too small under the Gaussian assumption.

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here

# Part 1d: Companion mass uncertainty

d = load("p1d_mass_samples")
mass = d["M2_Msun"].to_numpy()

# Posterior percentiles
q16, q50, q84 = np.percentile(
    mass, [16, 50, 84]
)

# Other uncertainty estimates
sd = np.std(mass, ddof=1)
variance = sd**2
full_width = q84 - q16
sem = sd / np.sqrt(len(mass))

print(f"Median mass = {q50:.5f} Msun")
print(f"16th percentile = {q16:.5f}")
print(f"84th percentile = {q84:.5f}")

print(f"Lower uncertainty = {q50-q16:.5f} Msun")
print(f"Upper uncertainty = {q84-q50:.5f} Msun")

print(f"Variance = {variance:.5f} Msun^2")
print(f"Standard deviation = {sd:.5f} Msun")
print(f"68% interval full width = {full_width:.5f} Msun")
print(f"SD / sqrt(N) = {sem:.5f} Msun")

# Plot posterior distribution
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    mass,
    bins=65,
    density=True,
    alpha=0.8,
    label="Posterior samples"
)

ax.axvline(q16, linestyle="--", label="16th percentile")
ax.axvline(q50, linestyle="--", label="Median")
ax.axvline(q84, linestyle="--", label="84th percentile")

ax.set_xlabel("Companion mass (Msun)")
ax.set_ylabel("Posterior density")
ax.legend()

plt.tight_layout()
plt.show()


**ANSWER:**

Using the posterior samples, I found a median companion mass of 7.74 solar masses. The lower uncertainty is 1.79 solar masses, and the upper uncertainty is 2.26 solar masses.

The three proposed uncertainty estimates have different problems.

First, the variance is about 4.58, but variance has units of solar masses squared, so it cannot be directly used as a mass uncertainty.

Second, subtracting the 16th percentile from the 84th percentile gives about 4.05 solar masses. This is the total width of the central 68% interval, not the uncertainty on each side.

Third, dividing the standard deviation by the square root of the number of samples gives about 0.024 solar masses. This measures the Monte Carlo uncertainty of the sample mean, assuming independent samples, rather than the uncertainty in the companion mass itself.

Because the posterior distribution is asymmetric, I would report the median with separate lower and upper uncertainties.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here

# Part 1e: Slope and intercept correlation

d = load("p1e_slope_intercept_samples")

slope = d["slope"].to_numpy()
intercept = d["intercept"].to_numpy()

# Posterior means
mean_slope = np.mean(slope)
mean_intercept = np.mean(intercept)

# Covariance matrix
cov = np.cov(slope, intercept, ddof=1)

sigma_slope_marg = np.sqrt(cov[0, 0])
sigma_intercept = np.sqrt(cov[1, 1])

# Correlation coefficient
rho = cov[0, 1] / (
    sigma_slope_marg * sigma_intercept
)

# Conditional uncertainty for Gaussian posterior
sigma_slope_fixed = (
    sigma_slope_marg * np.sqrt(1 - rho**2)
)

ratio = sigma_slope_fixed / sigma_slope_marg

print(f"Mean slope = {mean_slope:.6f}")
print(f"Mean intercept = {mean_intercept:.6f}")
print(f"Correlation = {rho:.6f}")

print(f"Marginal uncertainty = {sigma_slope_marg:.6f}")
print(f"Fixed-intercept uncertainty = {sigma_slope_fixed:.6f}")
print(f"Conditional / Marginal = {ratio:.4f}")

# Plot posterior samples
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    intercept,
    slope,
    s=3,
    alpha=0.12,
    label="Posterior samples"
)

ax.errorbar(
    mean_intercept,
    mean_slope,
    yerr=sigma_slope_marg,
    fmt="o",
    color="C1",
    capsize=5,
    label="Marginal uncertainty"
)

ax.errorbar(
    mean_intercept + 0.20,
    mean_slope,
    yerr=sigma_slope_fixed,
    fmt="s",
    color="C3",
    capsize=5,
    label="Conditional uncertainty"
)

ax.set_xlabel("Intercept")
ax.set_ylabel("Slope")
ax.legend()

plt.tight_layout()
plt.show()


**ANSWER:**

The marginal uncertainty in the slope is 0.02858, while the uncertainty obtained by fixing the intercept is 0.01667.

The slope and intercept are strongly negatively correlated, with a correlation coefficient of approximately -0.8122.

Assuming a Gaussian distribution for the two parameters, the conditional uncertainty is calculated by multiplying the marginal uncertainty by the square root of one minus the correlation coefficient squared. This gives a conditional uncertainty about 58.3% as large as the marginal uncertainty.

The fixed-intercept uncertainty is smaller because fixing the intercept removes part of the uncertainty caused by the correlation between the two parameters.

However, if the intercept is not known exactly, the marginal uncertainty of 0.02858 is more appropriate. Fixing the intercept in that situation would underestimate the actual uncertainty in the slope.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy import stats
from astropy.cosmology import FlatLambdaCDM

# Load Part 2 data
d2 = load("p2_pantheon_sne")

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

z = d2["z"].to_numpy()
mB = d2["mB"].to_numpy()
x1 = d2["x1"].to_numpy()
color = d2["c"].to_numpy()
host = d2["host_logmass"].to_numpy()

mu = cosmo.distmod(z).value
high_mass = (host > 10).astype(float)

# Peculiar velocity uncertainty, in magnitudes
sigma_pec = (5 / np.log(10)) * 250 / (299792.458 * z)

print("Number of supernovae:", len(d2))
print(d2.head())



In [ ]:
# 2a: your work here

# Part 2a: Hubble residuals

residual = mB - mu

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].scatter(x1, residual, alpha=0.6, s=15)
axes[0, 0].set_xlabel("Stretch (x1)")
axes[0, 0].set_ylabel("mB - mu (mag)")
axes[0, 0].set_title("Hubble residual vs stretch")

axes[0, 1].scatter(color, residual, alpha=0.6, s=15)
axes[0, 1].set_xlabel("Color (c)")
axes[0, 1].set_ylabel("mB - mu (mag)")
axes[0, 1].set_title("Hubble residual vs color")

axes[1, 0].scatter(host, residual, alpha=0.6, s=15)
axes[1, 0].axvline(10, color="red", linestyle="--")
axes[1, 0].set_xlabel("Host log mass")
axes[1, 0].set_ylabel("mB - mu (mag)")
axes[1, 0].set_title("Hubble residual vs host mass")

axes[1, 1].scatter(host, x1, alpha=0.6, s=15)
axes[1, 1].axvline(10, color="red", linestyle="--")
axes[1, 1].set_xlabel("Host log mass")
axes[1, 1].set_ylabel("Stretch (x1)")
axes[1, 1].set_title("Stretch vs host mass")

plt.tight_layout()
plt.show()

# Numerical trends
print("Correlation with stretch:",
      np.corrcoef(x1, residual)[0, 1])

print("Correlation with color:",
      np.corrcoef(color, residual)[0, 1])

print("Correlation of host mass and stretch:",
      np.corrcoef(host, x1)[0, 1])


**ANSWER:**

The Hubble residual is calculated by subtracting the distance modulus from the observed apparent magnitude.

According to the Tripp relation, the expected trend with stretch is negative because supernovae with broader light curves are intrinsically brighter. The expected trend with color is positive because redder supernovae tend to be fainter.

The raw trend with host mass does not necessarily show the true host-mass step. This is because host mass may also be correlated with stretch and color. We need to correct for both of these effects before measuring the mass step.

Some supernovae appear much fainter than the main population, suggesting possible contaminants. I will identify these more carefully using standardized residuals before fitting the final model.

Both alpha and beta have units of magnitudes per unit of their respective dimensionless light-curve parameters.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here

# Part 2b: Likelihood and priors

mB_err = d2["mB_err"].to_numpy()
x1_err = d2["x1_err"].to_numpy()
c_err = d2["c_err"].to_numpy()

cov_mB_x1 = d2["cov_mB_x1"].to_numpy()
cov_mB_c = d2["cov_mB_c"].to_numpy()
cov_x1_c = d2["cov_x1_c"].to_numpy()

# Parameters:
# theta = [M, alpha, beta, gamma, log_sigma_int]

def total_variance(theta):
    M, alpha, beta, gamma, log_sigma = theta
    sigma_int = np.exp(log_sigma)

    var = (
        mB_err**2
        + alpha**2 * x1_err**2
        + beta**2 * c_err**2
        + 2 * alpha * cov_mB_x1
        - 2 * beta * cov_mB_c
        - 2 * alpha * beta * cov_x1_c
        + sigma_pec**2
        + sigma_int**2
    )

    return var

def model_prediction(theta):
    M, alpha, beta, gamma, log_sigma = theta

    return (
        M + mu
        - alpha * x1
        + beta * color
        + gamma * high_mass
    )

# Start with a broad robust fit to find contaminants
A = np.column_stack([
    np.ones(len(mB)), -x1, color, high_mass
])

initial = np.linalg.lstsq(A, mB - mu, rcond=None)[0]
prelim_resid = (mB - mu) - A @ initial

print("Largest preliminary residuals:")
order = np.argsort(prelim_resid)[::-1]

for idx in order[:10]:
    print(d2["name"].iloc[idx],
          round(prelim_resid[idx], 3), "mag")

# Predefined conservative cut for extreme faint outliers
# Review the printed objects before accepting this cut.
keep = prelim_resid < 0.8

print("Objects removed:", np.sum(~keep))
print("Objects retained:", np.sum(keep))

# Log likelihood for selected objects
def log_likelihood(theta, mask=keep):
    var = total_variance(theta)

    if np.any(var <= 0):
        return -np.inf

    r = mB - model_prediction(theta)

    return -0.5 * np.sum(
        r[mask]**2 / var[mask]
        + np.log(2 * np.pi * var[mask])
    )

# Uniform prior ranges
bounds = [
    (-20.5, -18.0),             # M
    (0.0, 0.5),                 # alpha
    (0.0, 6.0),                 # beta
    (-0.3, 0.3),                # gamma
    (np.log(0.01), np.log(0.5)) # log sigma_int
]

def log_prior(theta):
    for value, (lower, upper) in zip(theta, bounds):
        if not lower < value < upper:
            return -np.inf
    return 0.0

def log_posterior(theta):
    lp = log_prior(theta)

    if not np.isfinite(lp):
        return -np.inf

    return lp + log_likelihood(theta)

theta_start = np.array([
    -19.3, 0.14, 3.1, -0.06, np.log(0.12)
])

print("Initial log likelihood:",
      log_likelihood(theta_start))


**ANSWER:**

I used a Gaussian likelihood for the supernova magnitudes, including the SALT2 measurement covariance, peculiar velocity uncertainty, and intrinsic scatter.

The measurement variance is calculated as:

Var = mB_err² + alpha² × x1_err² + beta² × c_err² + 2 × alpha × Cov(mB,x1) - 2 × beta × Cov(mB,c) - 2 × alpha × beta × Cov(x1,c).

The off-diagonal terms are necessary because the light-curve parameters are measured together and their errors are correlated.

The Gaussian log-likelihood includes both the squared residual divided by the total variance and the logarithm of the total variance. The logarithmic term is important because the variance itself depends on the model parameters.

I used uniform priors for M between -20.5 and -18.0 mag, alpha between 0 and 0.5, beta between 0 and 6, and gamma between -0.3 and 0.3 mag. I used a uniform prior on log intrinsic scatter between log(0.01) and log(0.5). These ranges allow physically plausible values without placing a strong preference near the expected solution.

I sampled log intrinsic scatter because this guarantees a positive scatter and allows the sampler to explore fractional changes more naturally.

For contaminants, I inspected preliminary standardized-model residuals and used a conservative cut to identify extremely faint outliers. Removing objects reduces the sample size and could bias the result if genuine supernovae are incorrectly excluded. Therefore, the fitted results should also be checked for sensitivity to this decision.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# Part 2c - Cell 1: MAP estimation

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

names = ["M", "alpha", "beta", "gamma", "log_sigma"]

# Find MAP
result = minimize(
    lambda t: -log_posterior(t),
    theta_start,
    method="Nelder-Mead",
    options={"maxiter": 20000,
             "xatol": 1e-9,
             "fatol": 1e-9}
)

theta_map = result.x

print("MAP optimization success:", result.success)

for name, value in zip(names, theta_map):
    print(f"{name} = {value:.6f}")

print(f"sigma_int = {np.exp(theta_map[4]):.6f} mag")


# Numerical Hessian
def hessian(f, x, eps=1e-4):
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n))

    for i in range(n):
        ei = np.zeros(n)
        ei[i] = eps

        for j in range(n):
            ej = np.zeros(n)
            ej[j] = eps

            H[i, j] = (
                f(x + ei + ej)
                - f(x + ei - ej)
                - f(x - ei + ej)
                + f(x - ei - ej)
            ) / (4 * eps**2)

    return (H + H.T) / 2


H_map = hessian(
    lambda t: -log_likelihood(t),
    theta_map
)

cov_map = np.linalg.inv(H_map)
map_errors = np.sqrt(np.diag(cov_map))

print("\nApproximate MAP uncertainties:")

for name, error in zip(names, map_errors):
    print(f"{name}: {error:.6f}")


In [ ]:

# Part 2c - Cell 2: Four independent MCMC chains

rng = np.random.default_rng(457)

ndim = 5
n_chains = 4
n_steps = 15000
burn = 5000

# Symmetrize and regularize the MAP covariance
eigval, eigvec = np.linalg.eigh(
    (cov_map + cov_map.T) / 2
)

cov_map = (
    eigvec
    @ np.diag(np.maximum(eigval, 1e-10))
    @ eigvec.T
)

# Proposal covariance
proposal_cov = (2.38**2 / ndim) * cov_map
L = np.linalg.cholesky(proposal_cov)

# Matrix used to spread the initial positions
start_L = np.linalg.cholesky(cov_map)

def run_chain(start, n_steps, rng):
    chain = np.zeros((n_steps, ndim))

    current = start.copy()
    current_lp = log_posterior(current)

    accepted = 0

    for i in range(n_steps):

        proposal = current + L @ rng.normal(size=ndim)
        proposal_lp = log_posterior(proposal)

        if np.log(rng.random()) < proposal_lp - current_lp:
            current = proposal
            current_lp = proposal_lp
            accepted += 1

        chain[i] = current

    return chain, accepted / n_steps


all_chains = []
acceptance = []
starting_points = []

for k in range(n_chains):

    chain_rng = np.random.default_rng(457 + k)

    # Start farther away than the typical posterior width
    for attempt in range(10000):
        direction = chain_rng.normal(size=ndim)
        direction /= np.linalg.norm(direction)

        distance = chain_rng.uniform(4, 8)

        start = (
            theta_map
            + distance * start_L @ direction
        )

        if np.isfinite(log_posterior(start)):
            break
    else:
        raise RuntimeError("Could not find a valid chain start")

    starting_points.append(start.copy())

    chain, acc = run_chain(
        start, n_steps, chain_rng
    )

    all_chains.append(chain)
    acceptance.append(acc)

chains = np.array(all_chains)

# Discard burn-in; retain unthinned draws for ESS
samples = chains[:, burn:, :]

print("Samples shape:", samples.shape)
print("Acceptance fractions:", acceptance)

print("\nStarting points:")
for k, start in enumerate(starting_points):
    print(f"Chain {k+1}:", np.round(start, 4))


In [ ]:

# Part 2c - Cell 3: Convergence diagnostics

# Split each chain into two halves
def split_chains(x):
    n = x.shape[1]
    half = n // 2

    return np.concatenate(
        [x[:, :half, :], x[:, -half:, :]],
        axis=0
    )

split = split_chains(samples)

# Traditional split R-hat
def calculate_rhat(x):
    m, n, p = x.shape

    chain_means = np.mean(x, axis=1)
    chain_vars = np.var(x, axis=1, ddof=1)

    W = np.mean(chain_vars, axis=0)
    B = n * np.var(chain_means, axis=0, ddof=1)

    var_hat = (n - 1) / n * W + B / n

    return np.sqrt(var_hat / W)

rhat = calculate_rhat(split)


# Autocorrelation-based ESS using FFT
def calculate_ess(x):
    m, n = x.shape

    centered = x - np.mean(x, axis=1, keepdims=True)

    # FFT autocovariance
    nfft = 1 << (2*n - 1).bit_length()

    fft = np.fft.rfft(centered, n=nfft, axis=1)

    acov = np.fft.irfft(
        fft * np.conjugate(fft),
        n=nfft,
        axis=1
    )[:, :n]

    acov /= np.arange(n, 0, -1)[None, :]

    W = np.mean(acov[:, 0])

    B = n * np.var(
        np.mean(x, axis=1), ddof=1
    )

    var_plus = (n - 1) / n * W + B / n

    rho = 1 - (
        W - np.mean(acov, axis=0)
    ) / var_plus

    # Initial positive paired sequence
    tau = -1.0

    for lag in range(0, n-1, 2):
        pair = rho[lag] + rho[lag+1]

        if pair <= 0:
            break

        tau += 2 * pair

    ess = m * n / max(tau, 1.0)

    return min(ess, m*n)


ess = np.array([
    calculate_ess(split[:, :, j])
    for j in range(ndim)
])

print("Convergence diagnostics:")
print(f"{'Parameter':<13} {'R-hat':>10} {'ESS':>12}")

for j, name in enumerate(names):
    print(f"{name:<13} {rhat[j]:>10.4f} {ess[j]:>12.1f}")


# Trace plots
fig, axes = plt.subplots(
    ndim, 1,
    figsize=(11, 12),
    sharex=True
)

for j, name in enumerate(names):

    for k in range(n_chains):

        axes[j].plot(
            chains[k, :, j],
            alpha=0.65,
            linewidth=0.45,
            label=f"Chain {k+1}"
        )

    axes[j].axvline(
        burn,
        color="black",
        linestyle="--"
    )

    axes[j].set_ylabel(name)

axes[0].legend()
axes[-1].set_xlabel("Iteration")

plt.tight_layout()
plt.show()


In [ ]:

# Part 2c - Cell 4: Posterior summaries

flat = samples.reshape(-1, ndim)

print("Posterior medians and 68% intervals:\n")

for j, name in enumerate(names):

    q16, q50, q84 = np.percentile(
        flat[:, j],
        [16, 50, 84]
    )

    print(
        f"{name} = {q50:.5f} "
        f"+{q84-q50:.5f} "
        f"-{q50-q16:.5f}"
    )


# Convert log_sigma to physical intrinsic scatter
sigma_samples = np.exp(flat[:, 4])

q16, q50, q84 = np.percentile(
    sigma_samples,
    [16, 50, 84]
)

print(
    f"\nsigma_int = {q50:.5f} "
    f"+{q84-q50:.5f} "
    f"-{q50-q16:.5f} mag"
)


In [ ]:

# Part 2c - Cell 5: Corner-style plot

plot_data = flat.copy()
plot_data[:, 4] = np.exp(plot_data[:, 4])

plot_names = [
    "M", "alpha", "beta",
    "gamma", "sigma_int"
]

fig, axes = plt.subplots(
    5, 5,
    figsize=(12, 12)
)

for i in range(5):
    for j in range(5):

        ax = axes[i, j]

        if i < j:
            ax.axis("off")

        elif i == j:
            ax.hist(
                plot_data[:, i],
                bins=35,
                density=True,
                alpha=0.8
            )

            q16, q50, q84 = np.percentile(
                plot_data[:, i],
                [16, 50, 84]
            )

            ax.axvline(q50, color="black")
            ax.axvline(q16, linestyle="--", color="gray")
            ax.axvline(q84, linestyle="--", color="gray")

        else:
            ax.scatter(
                plot_data[::5, j],
                plot_data[::5, i],
                s=2,
                alpha=0.12,
                rasterized=True
            )

        if i == 4 and j <= i:
            ax.set_xlabel(plot_names[j])

        if j == 0 and i > 0:
            ax.set_ylabel(plot_names[i])

        if i != 4:
            ax.set_xticklabels([])

        if j != 0:
            ax.set_yticklabels([])

plt.tight_layout()
plt.show()


# Correlation matrix
corr = np.corrcoef(flat, rowvar=False)

print("Correlation matrix:")

print(pd.DataFrame(
    corr,
    index=names,
    columns=names
))

# Identify strongest pair
pairs = []

for i in range(ndim):
    for j in range(i):
        pairs.append(
            (abs(corr[i, j]),
             names[i], names[j],
             corr[i, j])
        )

pairs.sort(reverse=True)

print("\nStrongest parameter correlations:")

for strength, name1, name2, value in pairs[:5]:
    print(f"{name1} and {name2}: r = {value:.4f}")


In [ ]:

# Part 2c - Cell 6: Fix M and refit gamma

fixed_M = theta_map[0]

# Remaining parameters:
# [alpha, beta, gamma, log_sigma]

def fixed_M_nll(p):
    theta = np.concatenate(
        ([fixed_M], p)
    )

    lp = log_posterior(theta)

    if not np.isfinite(lp):
        return 1e100

    return -lp


# Refit with fixed M
fit_fixed = minimize(
    fixed_M_nll,
    theta_map[1:],
    method="Nelder-Mead",
    options={"maxiter": 20000}
)

print("Fixed-M optimization success:", fit_fixed.success)

p_fixed = fit_fixed.x
gamma_fixed = p_fixed[2]

# Hessian for four free parameters
H_fixed = hessian(
    fixed_M_nll,
    p_fixed
)

cov_fixed = np.linalg.inv(H_fixed)

# Gamma is parameter index 2
sigma_gamma_fixed = np.sqrt(cov_fixed[2, 2])

# Marginal uncertainty from original MCMC
sigma_gamma_marg = np.std(
    flat[:, 3], ddof=1
)

ratio = sigma_gamma_fixed / sigma_gamma_marg

print(f"M fixed at: {fixed_M:.5f} mag")

print(f"Refitted gamma: {gamma_fixed:.5f} mag")

print(
    f"Conditional sigma_gamma: "
    f"{sigma_gamma_fixed:.5f} mag"
)

print(
    f"Marginal sigma_gamma: "
    f"{sigma_gamma_marg:.5f} mag"
)

print(f"Conditional / Marginal ratio: {ratio:.4f}")


# Compare the uncertainties visually
fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    0,
    np.median(flat[:, 3]),
    yerr=sigma_gamma_marg,
    fmt="o",
    capsize=6,
    label="M marginalized"
)

ax.errorbar(
    1,
    gamma_fixed,
    yerr=sigma_gamma_fixed,
    fmt="o",
    capsize=6,
    label="M fixed"
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Marginal", "Fixed M"])
ax.set_ylabel("Gamma (mag)")
ax.set_title("Comparison of gamma uncertainties")

plt.tight_layout()
plt.show()


In [ ]:

# Part 2c - Cell 7: Final numerical summary

print("===== PART 2C SUMMARY =====")

print("\nMAP parameters:")
for name, value in zip(names, theta_map):
    print(f"{name}: {value:.5f}")

print("\nConvergence:")
for j, name in enumerate(names):
    print(
        f"{name}: Rhat={rhat[j]:.4f}, "
        f"ESS={ess[j]:.0f}"
    )

print("\nPosterior parameters:")
for j in [1, 2, 3]:
    q16, q50, q84 = np.percentile(
        flat[:, j], [16, 50, 84]
    )

    print(
        f"{names[j]}: {q50:.5f} "
        f"+{q84-q50:.5f} "
        f"-{q50-q16:.5f}"
    )

q16, q50, q84 = np.percentile(
    sigma_samples, [16, 50, 84]
)

print(
    f"sigma_int: {q50:.5f} "
    f"+{q84-q50:.5f} "
    f"-{q50-q16:.5f} mag"
)

print("\nStrongest correlation:")
print(pairs[0])

print("\nGamma uncertainty comparison:")
print(f"Marginal: {sigma_gamma_marg:.5f} mag")
print(f"Fixed M:  {sigma_gamma_fixed:.5f} mag")
print(f"Ratio:    {ratio:.4f}")


**ANSWER:**

I first found the MAP parameters by minimizing the negative log-posterior. The MAP estimates were M = -19.2543 mag, alpha = 0.14593, beta = 2.86887, gamma = -0.06152 mag, and intrinsic scatter = 0.1079 mag.

I then used a Random-Walk Metropolis sampler with four independent chains. Each chain contained 15,000 iterations, and I discarded the first 5,000 as burn-in. I initialized the chains approximately 4–8 posterior standard deviations away from the MAP point, using the local covariance matrix to determine the directions.

To check convergence, I calculated the traditional split R-hat and autocorrelation-based effective sample size (ESS) for each parameter. The largest R-hat was **1.0023** (log_sigma), and the smallest ESS was **2259** (gamma). All R-hat values were below 1.01, and all ESS values exceeded 2000, suggesting good convergence. I would run longer if R-hat exceeded 1.01, ESS were too small, or the trace plots showed poor mixing.

Using the posterior samples, I obtained the following median estimates and 16th–84th percentile uncertainties:

- Alpha: 0.14594 (+0.00955 / -0.00937)
- Beta: 2.87286 (+0.09957 / -0.10616)
- Gamma: -0.06166 (+0.01797 / -0.01855) mag
- Intrinsic scatter: 0.10889 (+0.00824 / -0.00794) mag

From the corner plot and correlation matrix, the strongest correlation was between **gamma and M**, with a correlation coefficient of **-0.7642**. This negative correlation means that increasing one parameter can partly compensate for decreasing the other when fitting the observed supernova magnitudes.

Finally, I fixed M at its MAP value and refitted the other parameters. The resulting approximate conditional uncertainty in gamma was **0.01201 mag**, compared with a marginal uncertainty of **0.01861 mag** from the original MCMC posterior. The conditional uncertainty is about **64.5%** of the marginal uncertainty.

The marginal uncertainty is more appropriate when M is not known exactly, because it accounts for the additional uncertainty from allowing M to vary.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here

# Part 2d: Model comparison

# Model 1: gamma is free
fit1 = minimize(
    lambda t: -log_likelihood(t),
    theta_map,
    method="Nelder-Mead",
    options={"maxiter": 10000}
)

theta1 = fit1.x
logL1 = log_likelihood(theta1)

# Model 0: gamma fixed to zero
def theta0_from_free(p):
    return np.array([p[0], p[1], p[2], 0.0, p[3]])

def nll0(p):
    return -log_likelihood(theta0_from_free(p))

p0_start = np.array([
    theta_map[0], theta_map[1],
    theta_map[2], theta_map[4]
])

fit0 = minimize(
    nll0,
    p0_start,
    method="Nelder-Mead",
    options={"maxiter": 10000}
)

theta0 = theta0_from_free(fit0.x)
logL0 = log_likelihood(theta0)

# BIC
N = np.sum(keep)

bic0 = 4 * np.log(N) - 2 * logL0
bic1 = 5 * np.log(N) - 2 * logL1

delta_bic = bic0 - bic1

# Hessians and Laplace approximation
H1 = hessian(
    lambda t: -log_likelihood(t), theta1
)
H0 = hessian(nll0, fit0.x)

sign1, logdetH1 = np.linalg.slogdet(H1)
sign0, logdetH0 = np.linalg.slogdet(H0)

if sign0 <= 0 or sign1 <= 0:
    raise ValueError("Hessian not positive definite.")

# Gamma prior width = 0.6 mag
W_gamma = 0.6

two_log_B10 = (
    2 * (logL1 - logL0)
    + np.log(2 * np.pi)
    - logdetH1 + logdetH0
    - 2 * np.log(W_gamma)
)

# Twice as wide prior
two_log_B10_wide = two_log_B10 - 2*np.log(2)

gamma_fit = theta1[3]
sigma_gamma = np.std(flat[:, 3], ddof=1)

print("logL0 =", logL0)
print("logL1 =", logL1)

print("BIC0 - BIC1 =", delta_bic)
print("2 ln B10 =", two_log_B10)
print("2 ln B10 (double prior width) =", two_log_B10_wide)

print("Best gamma =", gamma_fit, "mag")
print("Marginal gamma uncertainty =", sigma_gamma, "mag")

# Detectability of a published -0.06 mag step
print(
    "Detection significance of a -0.06 mag step:",
    0.06 / sigma_gamma, "sigma"
)


**ANSWER:**

I compared two models: M0, which fixes the host-mass step to zero, and M1, which allows gamma to vary.

For M1, I used a uniform prior on gamma between -0.3 and 0.3 mag, corresponding to a prior width of 0.6 mag. This range includes the published estimate of approximately -0.06 mag while allowing for substantially larger steps.

I calculated the Bayes factor using the Laplace approximation, which accounts for both the improvement in maximum likelihood and the extra parameter volume.

When the gamma prior is twice as wide, the value of 2 ln B10 decreases by approximately 1.39, provided the posterior remains well inside the prior boundaries. This is because a broader prior increases the Occam penalty for the more complicated model.

I also calculated BIC0 minus BIC1. A positive difference favors the model with a free gamma parameter, while a negative difference favors the model without the step.

I compared the fitted gamma with the published value of -0.06 mag. The ratio of 0.06 mag to the marginal gamma uncertainty gives an approximate measure of how significantly this dataset could detect a step of that size.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here

# Part 2e: Injection-recovery coverage test

rng = np.random.default_rng(2026)

n_sim = 40
true_gamma = -0.06

# Use fitted parameters as the simulation truth,
# but inject a known gamma.
theta_truth = theta1.copy()
theta_truth[3] = true_gamma

# Fixed-design simulation:
# keep redshift, stretch, color, and host mass fixed
true_mean = model_prediction(theta_truth)
true_variance = total_variance(theta_truth)

# Save the original observed magnitudes
original_mB = mB.copy()

recovered = []
recovered_errors = []
successes = []

# Define a failure before inspecting results:
# fewer than 50% or more than 85% coverage,
# or a mean recovery bias larger than 2 standard
# errors of the mean across simulations.

try:
    for j in range(n_sim):
        # Generate synthetic magnitudes
        fake_mB = true_mean + rng.normal(
            0, np.sqrt(true_variance)
        )

        # Change the data used by the likelihood
        mB[:] = fake_mB

        # Fit simulated dataset
        fit = minimize(
            lambda t: -log_likelihood(t),
            theta_truth,
            method="Nelder-Mead",
            options={"maxiter": 5000}
        )

        if not fit.success:
            continue

        theta_hat = fit.x

        # Curvature uncertainty
        H = hessian(
            lambda t: -log_likelihood(t),
            theta_hat
        )

        try:
            cov_fit = np.linalg.inv(H)
            gamma_var = cov_fit[3, 3]

            if gamma_var <= 0:
                continue

            gamma_error = np.sqrt(gamma_var)

        except np.linalg.LinAlgError:
            continue

        recovered.append(theta_hat[3])
        recovered_errors.append(gamma_error)

        covered = (
            abs(theta_hat[3] - true_gamma)
            <= gamma_error
        )

        successes.append(covered)

finally:
    # Always restore original data
    mB[:] = original_mB

recovered = np.array(recovered)
recovered_errors = np.array(recovered_errors)
successes = np.array(successes)

print("Successful fits:", len(recovered))

if len(recovered) > 1:
    coverage = np.mean(successes)
    bias = np.mean(recovered - true_gamma)

    bias_se = np.std(
        recovered - true_gamma, ddof=1
    ) / np.sqrt(len(recovered))

    print("True gamma =", true_gamma)
    print("Mean recovered gamma =", np.mean(recovered))
    print("Mean bias =", bias)
    print("Bias standard error =", bias_se)
    print("68% interval coverage =", coverage)

    failed = (
        coverage < 0.50
        or coverage > 0.85
        or abs(bias) > 2 * bias_se
    )

    print("Pipeline test failed:", failed)

    # Plot recovered gamma estimates
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.errorbar(
        np.arange(len(recovered)),
        recovered,
        yerr=recovered_errors,
        fmt="o",
        markersize=3,
        capsize=2
    )

    ax.axhline(
        true_gamma,
        color="red",
        linestyle="--",
        label="True gamma"
    )

    ax.set_xlabel("Simulation number")
    ax.set_ylabel("Recovered gamma (mag)")
    ax.set_title("Injection-recovery test")
    ax.legend()

    plt.tight_layout()
    plt.show()


**ANSWER:**

I tested the fitting pipeline using an injection-recovery simulation.

I generated 40 synthetic supernova datasets using the observed redshifts, stretch values, colors, and host masses. I injected a known host-mass step of -0.06 mag and added Gaussian noise according to the model's total variance.

Before running the test, I defined failure as a 68% interval coverage below 50% or above 85%, or a mean recovery bias greater than twice its estimated standard error.

For each simulated dataset, I fitted the model again and calculated the uncertainty in gamma using the curvature of the log-likelihood.

I then checked how often the estimated 68% interval included the true gamma value. For a correctly calibrated model under the simulation assumptions, the expected coverage is approximately 68%.

This test can detect some fitting or uncertainty-calculation problems. However, it does not fully test whether the contaminant-removal procedure works correctly, because the same selection of objects is used throughout the simulations.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here

# Part 3a: Referee report - data and model tests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm

# Use the load() function from Part 1
wd = load("p3_wd_masses")

mass = wd["mass_Msun"].to_numpy()
mass_err = wd["mass_err_Msun"].to_numpy()

# Original AI's 3-sigma clipping
median_mass = np.median(mass)
std_mass = np.std(mass)

keep = np.abs(mass - median_mass) < 3 * std_mass

print("Total stars:", len(mass))
print("Removed by AI:", np.sum(~keep))
print("Remaining:", np.sum(keep))

print("\nRemoved masses:")
print(np.round(mass[~keep], 3))

print("\nMeasurement errors:")
print("Median error:", np.median(mass_err), "Msun")
print("Maximum error:", np.max(mass_err), "Msun")


# Two-Gaussian mixture, including measurement errors
def mixture_nll(theta, x, xerr):
    mu1, s1, mu2, s2, f = theta

    if not (
        0.3 < mu1 < 1.3
        and 0.3 < mu2 < 1.4
        and mu1 < mu2
        and 0.005 < s1 < 0.5
        and 0.005 < s2 < 0.5
        and 0 < f < 1
    ):
        return 1e10

    # Observed variance = intrinsic + measurement
    sigma1 = np.sqrt(s1**2 + xerr**2)
    sigma2 = np.sqrt(s2**2 + xerr**2)

    logp1 = np.log(f) + norm.logpdf(
        x, mu1, sigma1
    )

    logp2 = np.log(1-f) + norm.logpdf(
        x, mu2, sigma2
    )

    return -np.sum(np.logaddexp(logp1, logp2))


# Multiple starting points help avoid local minima
def fit_mixture(x, xerr, seed=457):
    rng = np.random.default_rng(seed)

    best = None

    for k in range(12):
        start = np.array([
            rng.uniform(0.57, 0.64),
            rng.uniform(0.02, 0.09),
            rng.uniform(0.77, 0.95),
            rng.uniform(0.12, 0.25),
            rng.uniform(0.5, 0.8)
        ])

        result = minimize(
            mixture_nll,
            start,
            args=(x, xerr),
            method="Nelder-Mead",
            options={
                "maxiter": 3000,
                "xatol": 1e-7,
                "fatol": 1e-7
            }
        )

        if best is None or result.fun < best.fun:
            best = result

    return best.x, best.fun


# Case 1: AI's clipped dataset, no measurement errors
fit_ai, nll_ai = fit_mixture(
    mass[keep],
    np.zeros(np.sum(keep))
)

# Case 2: Full data, no measurement errors
fit_full_noerr, nll_full_noerr = fit_mixture(
    mass,
    np.zeros(len(mass))
)

# Case 3: Full data, include measurement errors
fit_correct, nll_correct = fit_mixture(
    mass,
    mass_err
)

# Case 4: Clipped data, include errors
fit_clip_err, nll_clip_err = fit_mixture(
    mass[keep],
    mass_err[keep]
)

param_names = ["mu1", "sigma1", "mu2", "sigma2", "f"]

print("\n=== MODEL COMPARISON ===")

for label, fit in [
    ("AI-style", fit_ai),
    ("Full, no errors", fit_full_noerr),
    ("Full, with errors", fit_correct),
    ("Clipped, with errors", fit_clip_err)
]:
    print(f"\n{label}")
    for name, value in zip(param_names, fit):
        print(f"{name} = {value:.5f}")

print("\nChange in high-mass population mean:")
print(
    f"{fit_correct[2] - fit_ai[2]:.5f} Msun"
)


In [ ]:

# Part 3a: Visual demonstrations

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Clipping effect
axes[0, 0].hist(
    mass[keep], bins=45,
    alpha=0.7, label="Retained"
)

axes[0, 0].hist(
    mass[~keep], bins=20,
    alpha=0.8, label="Removed"
)

axes[0, 0].set_xlabel("Mass (Msun)")
axes[0, 0].set_ylabel("Number of stars")
axes[0, 0].set_title("Effect of 3-sigma clipping")
axes[0, 0].legend()


# 2. Measurement errors versus mass
axes[0, 1].scatter(
    mass, mass_err, s=12, alpha=0.5
)

axes[0, 1].set_xlabel("Mass (Msun)")
axes[0, 1].set_ylabel("Mass uncertainty (Msun)")
axes[0, 1].set_title("Individual measurement errors")


# 3. Mixture fit comparison
xgrid = np.linspace(
    mass.min(), mass.max(), 500
)

def mixture_pdf(x, p):
    mu1, s1, mu2, s2, f = p

    return (
        f * norm.pdf(x, mu1, s1)
        + (1-f) * norm.pdf(x, mu2, s2)
    )

axes[1, 0].hist(
    mass, bins=45, density=True,
    alpha=0.4, label="All data"
)

axes[1, 0].plot(
    xgrid,
    mixture_pdf(xgrid, fit_ai),
    label="AI-style clipped fit"
)

axes[1, 0].plot(
    xgrid,
    mixture_pdf(xgrid, fit_correct),
    label="Full-data fit"
)

axes[1, 0].set_xlabel("Mass (Msun)")
axes[1, 0].set_ylabel("Density")
axes[1, 0].set_title("Effect on fitted distribution")
axes[1, 0].legend()


# 4. Gaussian component widths
labels = ["Low-mass width", "High-mass width"]

axes[1, 1].bar(
    np.arange(2) - 0.2,
    [fit_full_noerr[1], fit_full_noerr[3]],
    width=0.4, label="Ignore errors"
)

axes[1, 1].bar(
    np.arange(2) + 0.2,
    [fit_correct[1], fit_correct[3]],
    width=0.4, label="Include errors"
)

axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_ylabel("Intrinsic width (Msun)")
axes[1, 1].set_title("Effect of measurement errors")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


# Compare one Gaussian against two using same full data
def one_gaussian_nll(p):
    mu, sigma = p

    if sigma <= 0:
        return 1e10

    total_sigma = np.sqrt(
        sigma**2 + mass_err**2
    )

    return -np.sum(
        norm.logpdf(mass, mu, total_sigma)
    )

fit_one = minimize(
    one_gaussian_nll,
    [mass.mean(), mass.std()],
    method="Nelder-Mead"
)

N = len(mass)

bic_one = 2 * np.log(N) + 2 * fit_one.fun
bic_two = 5 * np.log(N) + 2 * nll_correct

print("One-component logL:", -fit_one.fun)
print("Two-component logL:", -nll_correct)
print("BIC (one component):", bic_one)
print("BIC (two components):", bic_two)
print("BIC_one - BIC_two:", bic_one - bic_two)


**REFEREE REPORT:**

## Referee Report

I reviewed the AI-generated white dwarf analysis and found several problems with its data selection, statistical model, uncertainty estimates, convergence diagnostics, and validation.

**1. Incorrect removal of high-mass white dwarfs**

The original analysis applies a 3-sigma clipping procedure before fitting the mass distribution. However, this is problematic because the distribution is not expected to be a single Gaussian. The high-mass tail is part of the population we are trying to study.

In my dataset of 600 white dwarfs, the clipping removes 9 stars, all with masses above approximately 1.15 solar masses.

I compared the two-Gaussian fits before and after clipping. The fitted mean of the high-mass population changes from approximately 0.7250 solar masses in the clipped sample to 0.7426 solar masses in the full sample when measurement errors are ignored.

This is a shift of about 0.018 solar masses. The clipping therefore changes the population we are trying to measure.

The better approach is to retain these objects unless there is independent evidence that their measurements are unreliable. Alternatively, a robust mixture model could explicitly account for contamination.

**2. Ignoring measurement uncertainties**

The original likelihood ignores the individual mass uncertainties, claiming they are much smaller than the population widths.

However, the median measurement uncertainty in my data is 0.012 solar masses, and the maximum is 0.069 solar masses. These uncertainties are not necessarily negligible for the narrow low-mass population.

I fitted the full dataset both with and without measurement errors.

Without measurement errors, the intrinsic width of the low-mass population was approximately 0.0348 solar masses. After including the measurement errors, it decreased to approximately 0.0308 solar masses.

The correct treatment is to add the measurement variance to the intrinsic variance of each Gaussian component. This accounts for the fact that the observed distribution is broadened by measurement noise.

**3. Confusing population width with parameter uncertainty**

The original notebook reports the Gaussian population standard deviations as uncertainties on the mean masses.

These are different quantities. A population width describes how much the individual white dwarf masses vary, while uncertainty in a population mean describes how precisely the mean is estimated.

For example, my corrected fit gives a low-mass population width of approximately 0.031 solar masses. This number describes the intrinsic scatter of that population, not the uncertainty in its estimated mean.

The correct approach is to use the posterior distribution of each parameter and report the median together with its 16th and 84th percentiles. A properly calibrated bootstrap could also estimate frequentist uncertainty.

**4. Insufficient convergence diagnostics**

The original analysis calculates R-hat by treating the individual walkers of a single emcee ensemble as independent chains.

However, ensemble walkers interact during sampling and cannot automatically be treated as independent chains for a standard R-hat calculation. The notebook also prints that all chains have converged without requiring the diagnostics to pass.

A better approach would be to run multiple independent ensembles from dispersed initial conditions and examine split R-hat, effective sample size, autocorrelation, and trace plots.

The original convergence claim is therefore not sufficiently supported by the checks performed.

**5. Inadequate pipeline validation**

The original analysis generates one simulated dataset from the fitted model, refits it once, and concludes that the pipeline is unbiased because the fitted parameters are close to their input values.

However, recovering the parameters from one simulation does not demonstrate that the estimator is unbiased or that its uncertainty intervals have correct coverage.

A stronger validation would generate many independent synthetic datasets, refit each one, and measure the average bias and the fraction of uncertainty intervals that contain the true input parameters.

The test should also include challenging cases, such as overlapping populations, measurement noise, and contamination.

**6. Model comparison**

The original notebook concludes that the two-population model is decisively better because it produces a larger maximum likelihood.

This comparison is incomplete because the two-population model has more free parameters.

I compared one- and two-component Gaussian models using BIC on the same full dataset with measurement uncertainties included. The BIC difference, defined as BIC for one component minus BIC for two components, was approximately 220.6.

This strongly favors the two-component statistical description, even after applying a complexity penalty.

However, identifying two statistical components does not prove that they correspond to two distinct physical formation channels.

**Conclusion**

The original AI analysis contains several substantive statistical problems despite producing reasonable-looking plots.

The most important are removing potentially genuine high-mass white dwarfs, ignoring measurement uncertainties, and confusing intrinsic population widths with uncertainties in fitted parameters.

My corrected maximum-likelihood analysis shows that these choices can change the inferred mass distribution. In particular, the estimated high-mass population mean increases when the excluded stars are restored.

The data support a two-component description, but reliable physical conclusions still require appropriate uncertainty estimates, convergence tests, and simulation-based validation.

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

## AI-Use Appendix

I used ChatGPT to help write and debug Python code, understand statistical concepts, and organize my written explanations throughout the midterm.

For Part 1, I used it to help calculate uncertainties and explain the differences between formal, conditional, and marginal error bars.

For Part 2, I used it to help construct the likelihood, implement the MCMC sampler, calculate convergence diagnostics, and interpret the posterior results.

I also used ChatGPT to help review the provided AI-generated white dwarf analysis in Part 3 and develop numerical comparisons using my assigned dataset.

ChatGPT made several mistakes or gave suggestions that required checking. Its initial file paths did not work on my local computer. It also recommended installing ArviZ, which upgraded NumPy and caused compatibility problems in my Anaconda environment. I addressed this by checking the error messages, restoring compatible package versions, and using NumPy-based convergence diagnostics instead.

I checked the numerical outputs against the data and reviewed the assumptions behind the calculations. For example, I compared the white dwarf fits with and without clipping and measurement uncertainties instead of simply accepting the original AI-generated conclusions.

Without checking the AI output, I might have submitted code that did not run in my environment or reported unsupported conclusions about statistical convergence and uncertainty. In particular, I might have confused the width of a white dwarf population with the uncertainty in its fitted mean, or accepted the removal of genuine high-mass stars without testing its effect.

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.